# RetinaNet model for snowpole detection with LiDar and RGB datasets

### Import Packages

In [45]:
import os
import pandas as pd
from pathlib import Path
from PIL import Image

In [13]:
!python setup.py build_ext --inplace

running build_ext


In [12]:
!pip uninstall keras-retinanet


Found existing installation: keras-retinanet 1.0.0
Uninstalling keras-retinanet-1.0.0:
  Would remove:
    /home/chrsjoha/.local/bin/retinanet-convert-model
    /home/chrsjoha/.local/bin/retinanet-debug
    /home/chrsjoha/.local/bin/retinanet-evaluate
    /home/chrsjoha/.local/bin/retinanet-train
    /home/chrsjoha/.local/lib/python3.10/site-packages/keras_retinanet-1.0.0.dist-info/*
    /home/chrsjoha/.local/lib/python3.10/site-packages/keras_retinanet/*
    /home/chrsjoha/.local/lib/python3.10/site-packages/tests/*
Proceed (Y/n)? ERROR: Operation cancelled by user
^C


In [1]:
!pip install . --user

Looking in indexes: https://pypi.org/simple, https://pypi.idi.ntnu.no
Processing /work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet
  Preparing metadata (setup.py) ... done
  Using cached Cython-3.0.12-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.3 kB)
  Preparing metadata (setup.py) ... done
Using cached Cython-3.0.12-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.6 MB)
  Created wheel for keras-retinanet: filename=keras_retinanet-1.0.0-cp310-cp310-linux_x86_64.whl size=208081 sha256=6c4ac6166ee247e930f04b92ab65ab286162598f54737043cdb4a5faf1b0a9f1
  Stored in directory: /home/chrsjoha/.cache/pip/wheels/a3/a2/60/1dd91bc5e9538795982b1422e30478a759715c1d27fbf475a4
  Created wheel for keras-resnet: filename=keras_resnet-0.2.0-py2.py3-none-any.whl size=20486 sha256=771d00e956228daafa4609be7804d980403c501bda8992d2418226d55561fdf1
  Stored in directory: /home/chrsjoha/.cache/pip/wheels/16/af/88/a668b279c5eadbe55dcaf6207f09059135166cefb09088bac

### Convert Yolo dataset format to RetinaNet format

In [6]:
# Create symlinks for RGB
!mkdir -p retinanet_data/rgb/images/train retinanet_data/rgb/labels/train
!ln -s /datasets/tdt4265/ad/open/Poles/rgb/images/train/* retinanet_data/rgb/images/train/
!ln -s /datasets/tdt4265/ad/open/Poles/rgb/labels/train/* retinanet_data/rgb/labels/train/

!mkdir -p retinanet_data/rgb/images/valid retinanet_data/rgb/labels/valid
!ln -s /datasets/tdt4265/ad/open/Poles/rgb/images/valid/* retinanet_data/rgb/images/valid/
!ln -s /datasets/tdt4265/ad/open/Poles/rgb/labels/valid/* retinanet_data/rgb/labels/valid/

!mkdir -p retinanet_data/rgb/images/test retinanet_data/rgb/labels/test
!ln -s /datasets/tdt4265/ad/open/Poles/rgb/images/test/* retinanet_data/rgb/images/test/

# Create symlinks for LIDAR
!mkdir -p retinanet_data/lidar/images/train retinanet_data/lidar/labels/train
!ln -s /datasets/tdt4265/ad/open/Poles/lidar/combined_color/train/* retinanet_data/lidar/images/train/
!ln -s /datasets/tdt4265/ad/open/Poles/lidar/labels/train/* retinanet_data/lidar/labels/train/

!mkdir -p retinanet_data/lidar/images/valid retinanet_data/lidar/labels/valid
!ln -s /datasets/tdt4265/ad/open/Poles/lidar/combined_color/valid/* retinanet_data/lidar/images/valid/
!ln -s /datasets/tdt4265/ad/open/Poles/lidar/labels/valid/* retinanet_data/lidar/labels/valid/

!mkdir -p retinanet_data/lidar/images/test retinanet_data/lidar/labels/test
!ln -s /datasets/tdt4265/ad/open/Poles/lidar/combined_color/test/* retinanet_data/lidar/images/test/


In [43]:
def convert_yolo_to_retinanet_csv(image_dir, label_dir, output_csv, class_csv, class_map):
    image_dir = Path(image_dir)
    label_dir = Path(label_dir)

    annotations = []

    for label_file in sorted(label_dir.glob('*.txt')):
        img_file = image_dir / (label_file.stem + '.PNG')
        if not img_file.exists():
            img_file = image_dir / (label_file.stem + '.png')
        if not img_file.exists():
            print(f"Image for {label_file.stem} not found, skipping.")
            continue

        with Image.open(img_file) as img:
            img_w, img_h = img.size

        with open(label_file, 'r') as f:
            lines = f.readlines()

        if not lines:
            annotations.append([str(img_file), '', '', '', '', ''])
            continue

        for line in lines:
            class_id, x_center, y_center, width, height = map(float, line.strip().split())
            class_id = int(class_id)
            class_name = class_map.get(class_id, f'class_{class_id}')
            x1 = (x_center - width / 2) * img_w
            y1 = (y_center - height / 2) * img_h
            x2 = (x_center + width / 2) * img_w
            y2 = (y_center + height / 2) * img_h
            annotations.append([str(img_file), int(x1), int(y1), int(x2), int(y2), class_name])

    # Save annotations CSV
    df = pd.DataFrame(annotations, columns=['image_path', 'x1', 'y1', 'x2', 'y2', 'class_name'])
    df.to_csv(output_csv, index=False, header=False)
    print(f"Saved {len(df)} annotations to {output_csv}")

    # Save class mapping CSV
    with open(class_csv, 'w') as f:
        for i, name in sorted(class_map.items()):
            f.write(f"{name},{i}\n")
    print(f"Saved class mapping to {class_csv}")



In [46]:
# Example usage
class_map = {
    0: 'pole',
    # Add more classes here if needed
}

convert_yolo_to_retinanet_csv(
    image_dir='/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/retinanet_data/rgb/images/train',
    label_dir='/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/retinanet_data/rgb/labels/train',
    output_csv='/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/annotations/rgb_train_annotations.csv',
    class_csv='/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/classes.csv',
    class_map=class_map
)

convert_yolo_to_retinanet_csv(
    image_dir='/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/retinanet_data/rgb/images/valid',
    label_dir='/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/retinanet_data/rgb/labels/valid',
    output_csv='/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/annotations/rgb_valid_annotations.csv',
    class_csv='/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/classes.csv',
    class_map=class_map
)

convert_yolo_to_retinanet_csv(
    image_dir='/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/retinanet_data/rgb/images/test',
    label_dir='/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/retinanet_data/rgb/labels/test',
    output_csv='/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/annotations/rgb_test_annotations.csv',
    class_csv='/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/classes.csv',
    class_map=class_map
)

convert_yolo_to_retinanet_csv(
    image_dir='/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/retinanet_data/lidar/images/train',
    label_dir='/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/retinanet_data/lidar/labels/train',
    output_csv='/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/annotations/lidar_train_annotations.csv',
    class_csv='/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/classes.csv',
    class_map=class_map
)

convert_yolo_to_retinanet_csv(
    image_dir='/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/retinanet_data/lidar/images/valid',
    label_dir='/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/retinanet_data/lidar/labels/valid',
    output_csv='/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/annotations/lidar_valid_annotations.csv',
    class_csv='/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/classes.csv',
    class_map=class_map
)

convert_yolo_to_retinanet_csv(
    image_dir='/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/retinanet_data/lidar/images/test',
    label_dir='/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/retinanet_data/lidar/labels/test',
    output_csv='/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/annotations/lidar_test_annotations.csv',
    class_csv='/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/classes.csv',
    class_map=class_map
)


Saved 392 annotations to /work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/annotations/rgb_train_annotations.csv
Saved class mapping to /work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/classes.csv
Saved 113 annotations to /work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/annotations/rgb_valid_annotations.csv
Saved class mapping to /work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/classes.csv
Saved 0 annotations to /work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/annotations/rgb_test_annotations.csv
Saved class mapping to /work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/classes.csv
Saved 2754 annotations to /work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/annotations/lidar_train_annotations.csv
Saved class mapping to /work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/classes.csv
Saved 789 annotations to /work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/annotations/lidar_valid_annotations.csv
Saved class 

### Training the model

In [42]:
!keras_retinanet/bin/train.py csv /work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/annotations/lidar_train_annotations.csv /work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/classes.csv



2025-05-01 12:15:37.263966: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-01 12:15:37.289099: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-01 12:15:37.705596: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
Traceback (most recent call last):
  File "/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/keras_retinanet/bin/train.py", line 553, in <module>
    main()
  File "/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/keras

In [37]:
!python -m keras_retinanet.bin.train \
  csv /work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/annotations/lidar_test_annotations.csv /work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/classes.csv \
  --val-annotations /work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/annotations/lidar_valid_annotations.csv \
  --backbone resnet50 \
  --batch-size 4 \
  --epochs 50 \
  --steps 1000 \
  --weights imagenet \
  --snapshot-path ./snapshots \
  --image-min-side 600 \
  --image-max-side 1000

2025-05-01 12:11:46.210272: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-01 12:11:46.235788: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-01 12:11:46.669785: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
usage: train.py [-h]
                [--snapshot SNAPSHOT | --imagenet-weights | --weights WEIGHTS | --no-weights]
                [--backbone BACKBONE] [--batch-size BATCH_SIZE] [--gpu GPU]
                [--multi-gpu MULTI_GPU] [--mult

In [38]:
!retinanet-train \
  csv annotations/lidar_test_annotations.csv classes.csv \
  --val-annotations annotations/lidar_valid_annotations.csv \
  --backbone resnet50 \
  --batch-size 4 \
  --epochs 50 \
  --steps 1000 \
  --weights imagenet \
  --snapshot-path ./snapshots \
  --image-min-side 600 \
  --image-max-side 1000


/bin/bash: line 1: retinanet-train: command not found


In [10]:
!which retinanet-train

/home/chrsjoha/.local/bin/retinanet-train


In [11]:
!python -c "import keras_retinanet; print(keras_retinanet.__file__)"


/work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/keras_retinanet/__init__.py


In [33]:
import keras_retinanet
print("keras_retinanet version used:", keras_retinanet.__file__)


keras_retinanet version used: /work/chrsjoha/SnowConeDetection/RetinaNet/keras-retinanet/keras_retinanet/__init__.py
